#  Few-Shot vs. Fine-Tuned Summarization Comparison

This notebook demonstrates how to compare the performance of few-shot prompting with instruction-tuned models vs. fine-tuned summarization models (like BART).

In [1]:
!pip install -q transformers datasets ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.2 MB/s eta 0:00:00


In [2]:
#  Sample: Simulated movie reviews
movie_reviews = """
The film was a visual masterpiece with stunning cinematography.
However, the plot felt slow and predictable.
Great performances by the lead actors.
The score complemented the tone perfectly.
It dragged a bit in the middle but the ending was satisfying.
"""

##  1. Few-Shot Prompted Summarization (FLAN-T5)

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import time

few_shot_prompt = f"""
Below are several movie reviews:
{movie_reviews}

Summarize the overall sentiment and main points in 3 sentences.
"""

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

inputs = tokenizer(
    few_shot_prompt,
    return_tensors="pt",
    max_length=512,
    padding="max_length",
    truncation=True
)

start_time = time.time()
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)
end_time = time.time()

few_shot_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Few-shot Prompted Summary:\n")
print(few_shot_summary)
print(f"FLAN-T5 Inference Time: {end_time - start_time:.2f} seconds")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Few-shot Prompted Summary:

It's a visually stunning film, but it's also a slow and predictable one.
FLAN-T5 Inference Time: 6.69 seconds


## 2. Fine-Tuned Summarization (BART)

In [4]:
from transformers import pipeline
import time

try:
    summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
    start_time = time.time()
    summary = summarizer(movie_reviews, max_length=100, min_length=30, do_sample=False)
    end_time = time.time()
    print(" Fine-tuned Summarizer Output:\n")
    print(summary[0]['summary_text'])
    print(f"BART Inference Time: {end_time - start_time:.2f} seconds")
except Exception as e:
    print(" Error during summarization:", e)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu
Your max_length is set to 100, but your input_length is only 57. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=28)


 Fine-tuned Summarizer Output:

It dragged a bit in the middle but the ending was satisfying. The film was a visual masterpiece with stunning cinematography. The plot felt slow and predictable.
BART Inference Time: 24.45 seconds


###  Comparison: Few-Shot vs. Fine-Tuned Summarizer

| Aspect | Few-Shot Prompting (FLAN-T5) | Fine-Tuned Summarizer (BART-CNN) |
|--------|-------------------------------|-----------------------------------|
| Setup  | Instruction-based prompt | Task-specific pre-trained weights |
| Flexibility | Works on any task with clever prompts | Performs best on summarization |
| Output Style | More generic or varied | Concise and on-topic |
| Customization | Requires good prompt design | Requires labeled data for training |
| Inference Time | Typically slower | Optimized for summarization |
